# AFS Ingest

Upload PDFs to `@AFS_STAGE`, run `PARSE_DOCUMENT()` on each, and write extracted text to `COMMON.PDF_STAGING`.

**How to upload PDFs:**
- In the Snowflake UI, use the Files tab of `@AUDITED_FINANCIALS.COMMON.AFS_STAGE` to drag-and-drop PDF files.
- Or from SnowSQL / Snowflake CLI: `PUT file:///path/to/file.pdf @AUDITED_FINANCIALS.COMMON.AFS_STAGE`

Then run this notebook to extract and stage text for any PDFs not yet in `PDF_STAGING`.

In [ ]:
import sys
sys.path.insert(0, '../src')

from afs.pdf_ingest import extract_and_stage
import json

In [ ]:
# List PDFs in stage
staged_files = session.sql(
    "LIST @AUDITED_FINANCIALS.COMMON.AFS_STAGE PATTERN='.*\\.pdf'"
).collect()

print(f'Files in stage: {len(staged_files)}')
for f in staged_files:
    print(' -', f[0])

In [ ]:
# Find files not yet in PDF_STAGING
already_staged = {
    row[0] for row in session.sql(
        "SELECT FILENAME FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING"
    ).collect()
}

pending = []
for f in staged_files:
    # Stage listing returns paths like: afs_stage/filename.pdf
    full_path = f[0]
    filename = full_path.split('/')[-1]
    if filename not in already_staged:
        pending.append(filename)

print(f'Pending extraction: {len(pending)}')
for name in pending:
    print(' -', name)

In [ ]:
# Extract and stage each pending PDF
from datetime import datetime

results = []
for filename in pending:
    print(f'\nExtracting: {filename} ...', flush=True)
    try:
        row = extract_and_stage(session, filename)
        session.sql("""
            INSERT INTO AUDITED_FINANCIALS.COMMON.PDF_STAGING
              (FILENAME, FILING_ID, TOTAL_PAGES, PAGE_TEXTS, STATUS)
            SELECT %s, %s, %s, PARSE_JSON(%s), 'pending'
        """, params=[
            row['filename'],
            row['filing_id'],
            row['total_pages'],
            json.dumps(row['page_texts']),
        ]).collect()
        results.append({'file': filename, 'pages': row['total_pages'], 'status': 'ok'})
        print(f'  -> {row["total_pages"]} pages, filing_id={row["filing_id"][:12]}...')
    except Exception as e:
        results.append({'file': filename, 'status': 'error', 'error': str(e)})
        print(f'  -> ERROR: {e}')

print(f'\nDone. {sum(1 for r in results if r["status"]=="ok")}/{len(results)} succeeded.')

In [ ]:
# Show pending filings ready to parse
import pandas as pd

df = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     ORDER BY EXTRACTED_AT DESC
""").to_pandas()

print(f'Total in PDF_STAGING: {len(df)}')
df